In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ==========================================
# 1. GENERATE THE RAW MULTI-TENANT LOG FILE
# ==========================================
np.random.seed(42)
num_records = 1000
tenants = ['Acme_Corp', 'Beta_Industries', 'Delta_Systems', 'Omega_Labs']
tiers = ['Basic', 'Professional', 'Enterprise']
statuses = ['Active', 'Active', 'Active', 'Churned']

generated_data = []
start_date = datetime(2025, 1, 1)

for i in range(1, num_records + 1):
    tenant = np.random.choice(tenants)
    tier = np.random.choice(tiers)
    status = np.random.choice(statuses)
    
    monthly_price = 49.00 if tier == 'Basic' else (149.00 if tier == 'Professional' else 499.00)
    days_offset = np.random.randint(0, 500)
    signup_date = start_date + timedelta(days=days_offset)
    api_calls = np.random.randint(500, 100000) if status == 'Active' else np.random.randint(10, 5000)
    storage_used_gb = np.random.uniform(1.5, 500.0) if tier != 'Basic' else np.random.uniform(0.1, 50.0)
    
    generated_data.append({
        "account_id": f"ACC_{10000 + i}",
        "tenant_id": tenant,
        "subscription_tier": tier,
        "monthly_recurring_revenue": monthly_price,
        "account_status": status,
        "api_requests_count": api_calls,
        "cloud_storage_used_gb": round(storage_used_gb, 2),
        "activation_date": signup_date.strftime("%Y-%m-%d")
    })

raw_saas_df = pd.DataFrame(generated_data)
raw_csv_path = "raw_saas_tenant_logs.csv"
raw_saas_df.to_csv(raw_csv_path, index=False)
print(f"💾 File saved successfully as: '{raw_csv_path}'")

# ==========================================
# 2. INGEST DATA AND ROUTE SECURELY
# ==========================================
class TenantDataRouter:
    def __init__(self, file_path):
        self.data = pd.read_csv(file_path)
        
    def get_tenant_data(self, requesting_tenant_id):
        # Strict isolation filter
        isolated_data = self.data[self.data['tenant_id'] == requesting_tenant_id]
        
        # Security leak check
        leaked_rows = isolated_data[isolated_data['tenant_id'] != requesting_tenant_id]
        if not leaked_rows.empty:
            raise SecurityError("CRITICAL: Cross-Tenant Data Isolation Breach!")
            
        return isolated_data

# Initialize router with our file
router = TenantDataRouter(raw_csv_path)

# Test execution for Beta Industries
beta_isolated_data = router.get_tenant_data("Beta_Industries")
print(f"🔒 Beta_Industries isolation active. Row count fetched: {len(beta_isolated_data)}")

💾 File saved successfully as: 'raw_saas_tenant_logs.csv'
🔒 Beta_Industries isolation active. Row count fetched: 243


In [2]:
import pandas as pd
import numpy as np

# 1. Load the Raw Asset File Generated Yesterday
raw_data_path = "raw_saas_tenant_logs.csv"
saas_pipeline_df = pd.read_csv(raw_data_path)

print("--- Initial Data Sanitation Inspection ---")
print(f"Total Rows Loaded: {saas_pipeline_df.shape[0]}")
print(f"Missing Values Check:\n{saas_pipeline_df.isnull().sum()}\n")

# 2. Simulate & Clean Data Pipeline Anomalies (Sanitation Phase)
# Let's ensure our activation date is correctly formatted as a datetime object
saas_pipeline_df['activation_date'] = pd.to_datetime(saas_pipeline_df['activation_date'])

# 3. Feature Engineering: Calculate Core SaaS Business Metrics
# A: Create a binary numeric flag for Churn Status (1 if Churned, 0 if Active)
saas_pipeline_df['is_churned'] = saas_pipeline_df['account_status'].apply(lambda x: 1 if x == 'Churned' else 0)

# B: Calculate Usage Intensity Indicator (API Requests per GB of Cloud Storage)
# Handles potential division by zero errors by adding a tiny epsilon value (1e-5)
saas_pipeline_df['usage_intensity'] = round(
    saas_pipeline_df['api_requests_count'] / (saas_pipeline_df['cloud_storage_used_gb'] + 1e-5), 2
)

# C: Segment Accounts based on High vs Low Platform Utilization Volume
storage_threshold = saas_pipeline_df['cloud_storage_used_gb'].median()
saas_pipeline_df['storage_tier_segment'] = saas_pipeline_df['cloud_storage_used_gb'].apply(
    lambda x: 'High_Utilization' if x > storage_threshold else 'Standard_Utilization'
)

# 4. Save the Refined/Cleaned Master File
cleaned_csv_path = "cleaned_saas_tenant_analytics.csv"
saas_pipeline_df.to_csv(cleaned_csv_path, index=False)

print("✅ Data Sanitation & Feature Engineering Pipeline Complete!")
print(f"💾 Processed analytical dataset saved as: '{cleaned_csv_path}'")
print("\n--- Processed Dataset Preview (Top 3 Rows) ---")
print(saas_pipeline_df[['account_id', 'tenant_id', 'is_churned', 'usage_intensity', 'storage_tier_segment']].head(3))

--- Initial Data Sanitation Inspection ---
Total Rows Loaded: 1000
Missing Values Check:
account_id                   0
tenant_id                    0
subscription_tier            0
monthly_recurring_revenue    0
account_status               0
api_requests_count           0
cloud_storage_used_gb        0
activation_date              0
dtype: int64

✅ Data Sanitation & Feature Engineering Pipeline Complete!
💾 Processed analytical dataset saved as: 'cleaned_saas_tenant_analytics.csv'

--- Processed Dataset Preview (Top 3 Rows) ---
  account_id      tenant_id  is_churned  usage_intensity  storage_tier_segment
0  ACC_10001  Delta_Systems           0          9799.73  Standard_Utilization
1  ACC_10002  Delta_Systems           0           613.74  Standard_Utilization
2  ACC_10003  Delta_Systems           0           144.63      High_Utilization


In [3]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

# 1. Force the notebook to look at your exact project directory
current_dir = os.getcwd()
print(f"Directory active: {current_dir}")

# 2. GENERATE RAW DATA
np.random.seed(42)
num_records = 1000
tenants = ['Acme_Corp', 'Beta_Industries', 'Delta_Systems', 'Omega_Labs']
tiers = ['Basic', 'Professional', 'Enterprise']
statuses = ['Active', 'Active', 'Active', 'Churned']

generated_data = []
start_date = datetime(2025, 1, 1)

for i in range(1, num_records + 1):
    tenant = np.random.choice(tenants)
    tier = np.random.choice(tiers)
    status = np.random.choice(statuses)
    
    monthly_price = 49.00 if tier == 'Basic' else (149.00 if tier == 'Professional' else 499.00)
    days_offset = np.random.randint(0, 500)
    signup_date = start_date + timedelta(days=days_offset)
    api_calls = np.random.randint(500, 100000) if status == 'Active' else np.random.randint(10, 5000)
    storage_used_gb = np.random.uniform(1.5, 500.0) if tier != 'Basic' else np.random.uniform(0.1, 50.0)
    
    generated_data.append({
        "account_id": f"ACC_{10000 + i}",
        "tenant_id": tenant,
        "subscription_tier": tier,
        "monthly_recurring_revenue": monthly_price,
        "account_status": status,
        "api_requests_count": api_calls,
        "cloud_storage_used_gb": round(storage_used_gb, 2),
        "activation_date": signup_date.strftime("%Y-%m-%d")
    })

# Save Raw File
raw_df = pd.DataFrame(generated_data)
raw_df.to_csv("raw_saas_tenant_logs.csv", index=False)
print("📦 Success: 'raw_saas_tenant_logs.csv' generated and saved!")

# 3. CLEAN & ENGINEER FEATURES
raw_df['activation_date'] = pd.to_datetime(raw_df['activation_date'])
raw_df['is_churned'] = raw_df['account_status'].apply(lambda x: 1 if x == 'Churned' else 0)
raw_df['usage_intensity'] = round(raw_df['api_requests_count'] / (raw_df['cloud_storage_used_gb'] + 1e-5), 2)

storage_threshold = raw_df['cloud_storage_used_gb'].median()
raw_df['storage_tier_segment'] = raw_df['cloud_storage_used_gb'].apply(
    lambda x: 'High_Utilization' if x > storage_threshold else 'Standard_Utilization'
)

# Save Cleaned File
raw_df.to_csv("cleaned_saas_tenant_analytics.csv", index=False)
print("✅ Success: 'cleaned_saas_tenant_analytics.csv' engineered and saved!")

Directory active: C:\Users\smmoh
📦 Success: 'raw_saas_tenant_logs.csv' generated and saved!
✅ Success: 'cleaned_saas_tenant_analytics.csv' engineered and saved!


In [4]:
import pandas as pd

# 1. Ingest the Cleaned Dataset Engineered on Day 2
cleaned_csv_path = "cleaned_saas_tenant_analytics.csv"
analytics_df = pd.read_csv(cleaned_csv_path)

print("--- Data Ingestion Verified ---")
print(f"Loaded {analytics_df.shape[0]} rows for performance aggregation.\n")

# 2. Build the Multi-Tenant Summary Aggregation Pipeline
# We group by 'tenant_id' to extract isolated operational footprints
tenant_summary = analytics_df.groupby('tenant_id').agg(
    total_accounts=('account_id', 'count'),
    total_mrr=('monthly_recurring_revenue', 'sum'),
    average_api_usage=('api_requests_count', 'mean'),
    total_storage_gb=('cloud_storage_used_gb', 'sum'),
    churned_accounts=('is_churned', 'sum')
).reset_index()

# 3. Compute Derived Executive KPIs per Tenant
# Calculate Churn Rate percentage for each corporate client
tenant_summary['churn_rate_pct'] = round(
    (tenant_summary['churned_accounts'] / tenant_summary['total_accounts']) * 100, 2
)

# Clean up averages formatting for presentation
tenant_summary['average_api_usage'] = round(tenant_summary['average_api_usage'], 0)
tenant_summary['total_mrr'] = round(tenant_summary['total_mrr'], 2)
tenant_summary['total_storage_gb'] = round(tenant_summary['total_storage_gb'], 2)

# 4. Export the Aggregated KPI Asset File
aggregated_csv_path = "tenant_executive_summary.csv"
tenant_summary.to_csv(aggregated_csv_path, index=False)

print("📈 Aggregation Pipeline Engine Executed Successfully!")
print(f"💾 Executive summary KPIs saved as: '{aggregated_csv_path}'")
print("\n--- Visualizing Final Day 3 Corporate Aggregation Grid ---")
print(tenant_summary)

--- Data Ingestion Verified ---
Loaded 1000 rows for performance aggregation.

📈 Aggregation Pipeline Engine Executed Successfully!
💾 Executive summary KPIs saved as: 'tenant_executive_summary.csv'

--- Visualizing Final Day 3 Corporate Aggregation Grid ---
         tenant_id  total_accounts  total_mrr  average_api_usage  \
0        Acme_Corp             260    59640.0            36958.0   
1  Beta_Industries             243    56907.0            37233.0   
2    Delta_Systems             264    56936.0            40184.0   
3       Omega_Labs             233    50767.0            37000.0   

   total_storage_gb  churned_accounts  churn_rate_pct  
0          41659.57                70           26.92  
1          43139.72                55           22.63  
2          44281.27                68           25.76  
3          36996.51                66           28.33  


In [5]:
# Let's print all columns to see the exact name of the time column
print(analytics_df.columns.tolist())

['account_id', 'tenant_id', 'subscription_tier', 'monthly_recurring_revenue', 'account_status', 'api_requests_count', 'cloud_storage_used_gb', 'activation_date', 'is_churned', 'usage_intensity', 'storage_tier_segment']


In [6]:
print("--- Day 4: Time-Based Feature Engineering ---")

# 1. Convert activation_date to a standard datetime object
analytics_df['activation_date'] = pd.to_datetime(analytics_df['activation_date'])

# 2. Extract structured date and hourly features from activation_date
analytics_df['activation_date_only'] = analytics_df['activation_date'].dt.date
analytics_df['activation_hour'] = analytics_df['activation_date'].dt.hour

print("✅ Success: Extracted time-based features from activation_date!")
print("📊 Pre-split timeline check:")
print(analytics_df[['activation_date', 'activation_date_only', 'activation_hour']].head())

--- Day 4: Time-Based Feature Engineering ---
✅ Success: Extracted time-based features from activation_date!
📊 Pre-split timeline check:
  activation_date activation_date_only  activation_hour
0      2025-04-17           2025-04-17                0
1      2026-04-04           2026-04-04                0
2      2025-09-15           2025-09-15                0
3      2026-03-20           2026-03-20                0
4      2025-12-11           2025-12-11                0


In [7]:
print("--- Day 5: Multi-Tenant Operational Metrics Isolation ---")

# Group data by tenant and activation hour to find peak onboarding or activity times
hourly_tenant_performance = analytics_df.groupby(['tenant_id', 'activation_hour']).agg(
    active_accounts=('account_id', 'count'),
    hourly_mrr=('monthly_recurring_revenue', 'sum'),
    average_api_calls=('api_requests_count', 'mean'),
    total_storage_used=('cloud_storage_used_gb', 'sum')
).reset_index()

# Clean up formatting for a professional presentation grid
hourly_tenant_performance['average_api_calls'] = round(hourly_tenant_performance['average_api_calls'], 0)
hourly_tenant_performance['hourly_mrr'] = round(hourly_tenant_performance['hourly_mrr'], 2)

print("✅ Success: Day 5 hourly tenant metrics isolated successfully!")
print(hourly_tenant_performance.head(10))

--- Day 5: Multi-Tenant Operational Metrics Isolation ---
✅ Success: Day 5 hourly tenant metrics isolated successfully!
         tenant_id  activation_hour  active_accounts  hourly_mrr  \
0        Acme_Corp                0              260     59640.0   
1  Beta_Industries                0              243     56907.0   
2    Delta_Systems                0              264     56936.0   
3       Omega_Labs                0              233     50767.0   

   average_api_calls  total_storage_used  
0            36958.0            41659.57  
1            37233.0            43139.72  
2            40184.0            44281.27  
3            37000.0            36996.51  


In [8]:
print("--- Day 6: Tenant Segmentation & Risk Profile Mapping ---")

# 1. Aggregate core financial and operational health metrics per tenant
tenant_risk_profiles = analytics_df.groupby('tenant_id').agg(
    total_mrr=('monthly_recurring_revenue', 'sum'),
    total_api_calls=('api_requests_count', 'sum'),
    total_accounts=('account_id', 'count'),
    churned_accounts=('is_churned', 'sum')
).reset_index()

# 2. Calculate the Churn Rate percentage for each tenant
tenant_risk_profiles['tenant_churn_rate_%'] = round(
    (tenant_risk_profiles['churned_accounts'] / tenant_risk_profiles['total_accounts']) * 100, 2
)

# 3. Classify Risk Tiers based on calculated churn concentration
def assign_risk_tier(rate):
    if rate == 0: return 'Low Risk'
    elif rate <= 25: return 'Moderate Risk'
    else: return 'High Risk/Critical'

tenant_risk_profiles['risk_segment'] = tenant_risk_profiles['tenant_churn_rate_%'].apply(assign_risk_tier)

print("✅ Success: Day 6 tenant health and risk profiles mapped successfully!")
print(tenant_risk_profiles.sort_values(by='total_mrr', ascending=False).head(10))

--- Day 6: Tenant Segmentation & Risk Profile Mapping ---
✅ Success: Day 6 tenant health and risk profiles mapped successfully!
         tenant_id  total_mrr  total_api_calls  total_accounts  \
0        Acme_Corp    59640.0          9609156             260   
2    Delta_Systems    56936.0         10608477             264   
1  Beta_Industries    56907.0          9047641             243   
3       Omega_Labs    50767.0          8620971             233   

   churned_accounts  tenant_churn_rate_%        risk_segment  
0                70                26.92  High Risk/Critical  
2                68                25.76  High Risk/Critical  
1                55                22.63       Moderate Risk  
3                66                28.33  High Risk/Critical  


In [9]:
print("--- Day 7: Pipeline Standardization & Data Export ---")

# 1. Merge the Day 6 Risk Tier classifications back into the main DataFrame for complete record context
analytics_df = analytics_df.merge(
    tenant_risk_profiles[['tenant_id', 'tenant_churn_rate_%', 'risk_segment']],
    on='tenant_id',
    how='left'
)

# 2. Export the final processed master dataset
analytics_df.to_csv('final_processed_saas_analytics.csv', index=False)

# 3. Export aggregated tenant profiles for high-level executive review
tenant_risk_profiles.to_csv('executive_tenant_risk_profiles.csv', index=False)

print("✅ Success: Pipeline execution complete! Final datasets saved to CSV.")
print("📁 Saved Assets: 'final_processed_saas_analytics.csv' & 'executive_tenant_risk_profiles.csv'")
print(f"📊 Final Dataset Shape: {analytics_df.shape}")

--- Day 7: Pipeline Standardization & Data Export ---
✅ Success: Pipeline execution complete! Final datasets saved to CSV.
📁 Saved Assets: 'final_processed_saas_analytics.csv' & 'executive_tenant_risk_profiles.csv'
📊 Final Dataset Shape: (1000, 15)
